# Chapter 11 &mdash; The Pumping Lemma for Context-Free Languages

**Concept 18 of the Chapter 11 decomposition:** *The Pumping Lemma for Context-Free Languages*

Long strings split as $uvxyz$ with $|vy|>0$, $|vxy|\le N$, and $uv^ixy^iz\in L$ for all $i$.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-CFL-Pumping-Lemma/Concept-CFL-Pumping-Lemma.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


> **IF** $L$ is context-free **THEN** there is $N$ such that every $w\in L$ with
> $|w|\ge N$ can be written $w = uvxyz$ with
> $|vy|>0$, $|vxy|\le N$, and $uv^ixy^iz \in L$ for all $i\ge0$.

**Two pumped pieces, not one.** The reason is the **parse tree**, not the state graph:
a long enough string forces a path from root to leaf that repeats a nonterminal $A$,
and the subtree between the two $A$s can be **repeated or removed**. Repeating it
duplicates whatever lies to the **left** ($v$) and to the **right** ($y$) of the inner
$A$ &mdash; hence two pieces, pumped **in lock-step**.

Everything else mirrors Chapter 4: it is one-way, so use it only to **disprove**, and
you must rule out **every** split.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### Finding a repeated nonterminal on a root-to-leaf path

In [ ]:
Dyck = mkg({'S': ["", "(S)", "SS"]})

def paths(t, acc=()):
    if isinstance(t, str):
        yield acc + (t,); return
    for c in t[1:]:
        yield from paths(c, acc + (t[0],))

def repeated_nt(t):
    for p in paths(t):
        seen = {}
        for i, x in enumerate(p):
            if x in seen: return (x, seen[x], i, p)
            seen[x] = i
    return None

### The five-way split, and the pump

In [ ]:
def splits5(w, N):
    out = []
    n = len(w)
    for i in range(n+1):
        for j in range(i, n+1):
            for k in range(j, n+1):
                for l in range(k, n+1):
                    u, v, x, y, z = w[:i], w[i:j], w[j:k], w[k:l], w[l:]
                    if not (v or y): continue
                    if len(v + x + y) > N: continue
                    out.append((u, v, x, y, z))
    return out

def pump(sp, i):
    u, v, x, y, z = sp
    return u + v*i + x + y*i + z

## 3. Tests

A long enough string forces a repeated nonterminal on some path.

In [ ]:
t = parse_trees(Dyck, '((()))', cap=1)[0]
r = repeated_nt(t)
print("repeated nonterminal :", r[0], " at depths", r[1], "and", r[2])
print("path :", r[3])
assert r is not None

The two pumped pieces come from the **left and right** of the inner subtree.

In [ ]:
print("outer A derives   v  [ inner A ]  y")
print("inner A derives            x")
print()
print("repeating the outer-to-inner segment gives  v^i x y^i --")
print("two pieces, pumped together.  A DFA's lasso gave only one.")

Pumping a Dyck string keeps it in the language.

In [ ]:
def balanced(s):
    d = 0
    for ch in s:
        d += 1 if ch == '(' else -1
        if d < 0: return False
    return d == 0

sp = ('', '(', '()', ')', '')          # u v x y z
for i in range(5):
    w = pump(sp, i)
    print("  i=%d : %-14r balanced? %s" % (i, w, balanced(w)))
    assert balanced(w)

$\{a^nb^n\}$ pumps too &mdash; it **is** context-free, so the lemma must hold.

In [ ]:
def in_anbn(s):
    k = len(s) - len(s.lstrip('a'))
    return s == 'a'*k + 'b'*(len(s)-k) and k == len(s)-k

sp = ('', 'a', 'ab', 'b', '')
for i in range(5):
    w = pump(sp, i)
    print("  i=%d : %-12r in a^n b^n? %s" % (i, w, in_anbn(w)))
    assert in_anbn(w)

And the lemma is satisfiable: some split works for every long member.

In [ ]:
N = 6
for w in ['aabb', 'aaabbb', 'aaaabbbb']:
    good = [sp for sp in splits5(w, N)
            if all(in_anbn(pump(sp, i)) for i in range(4))]
    print("%-10r : %d of %d splits pump" % (w, len(good), len(splits5(w, N))))
    assert good

## 4. Exercises


1. Why does the tree argument give **two** pumped pieces and the lasso only one?
2. What plays the role of $|Q|$ in choosing $N$ here?
3. State the contrapositive, with all four quantifiers flipped.

In [ ]:
# Your work for the exercises above.